## M3 — Índice F-of-M
Construimos el índice de atractivo territorial para rankear las 36.334 secciones censales.
F-of-M(s) = 0.30·Solvencia + 0.25·Demanda + 0.25·Soledad + 0.15·Concentración + 0.05·Futuro

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df     = pd.read_csv('dataset_maestro.csv',
                     dtype={'seccion_censal': str},
                     encoding='utf-8-sig')
df_muni = pd.read_csv('municipios_limpios.csv')

print(f"Secciones cargadas: {df.shape[0]:,}")
print(f"Municipios cargados: {df_muni.shape[0]:,}")

Secciones cargadas: 36,334
Municipios cargados: 8,124


### Paso 1 — Normalización min-max de las variables

In [11]:
def normalizar(serie):
    minimo = serie.min()
    maximo = serie.max()
    return (serie - minimo) / (maximo - minimo)

# Normalizar cada variable
df['n_renta']          = normalizar(df['renta_hogar_2024'])
df['n_vol_75plus']     = normalizar(df['vol_75plus'])
df['n_pct_75plus']     = normalizar(df['pct_75plus'])
df['n_ratio_viudez']   = normalizar(df['ratio_viudez'])
df['n_cohorte_futura'] = normalizar(df['cohorte_futura'])

print("Variables normalizadas (rango 0-1):")
cols_n = ['n_renta', 'n_vol_75plus', 'n_pct_75plus',
          'n_ratio_viudez', 'n_cohorte_futura']
print(df[cols_n].describe().round(3))

Variables normalizadas (rango 0-1):
         n_renta  n_vol_75plus  n_pct_75plus  n_ratio_viudez  n_cohorte_futura
count  35012.000     36334.000     36334.000       36334.000         36334.000
mean       0.258         0.051         0.191           0.071             0.042
std        0.097         0.028         0.105           0.028             0.024
min        0.000         0.000         0.000           0.000             0.000
25%        0.191         0.032         0.121           0.055             0.028
50%        0.254         0.049         0.176           0.067             0.040
75%        0.298         0.067         0.239           0.082             0.054
max        1.000         1.000         1.000           1.000             1.000


### Paso 2 — Construir el índice F-of-M

In [12]:
# Pesos originales del equipo
# Solvencia:     0.30
# Demanda:       0.25 (vol + pct al 50% cada uno)
# Soledad:       0.25
# Concentración: 0.15 → pendiente de datos externos
# Futuro:        0.05

# Versión actual sin Concentración — pesos redistribuidos proporcionalmente
W_SOLVENCIA = 0.35   # 0.30 + parte de 0.15
W_VOL       = 0.15   # mitad del componente demanda
W_PCT       = 0.15   # mitad del componente demanda
W_SOLEDAD   = 0.30   # 0.25 + parte de 0.15
W_FUTURO    = 0.05

print("Pesos aplicados:")
print(f"  Solvencia:  {W_SOLVENCIA}")
print(f"  Volumen:    {W_VOL}")
print(f"  Porcentaje: {W_PCT}")
print(f"  Soledad:    {W_SOLEDAD}")
print(f"  Futuro:     {W_FUTURO}")
print(f"  TOTAL:      {W_SOLVENCIA+W_VOL+W_PCT+W_SOLEDAD+W_FUTURO}")

Pesos aplicados:
  Solvencia:  0.35
  Volumen:    0.15
  Porcentaje: 0.15
  Soledad:    0.3
  Futuro:     0.05
  TOTAL:      1.0


### Paso 3 — Calcular la puntuación final

In [13]:
df['fofm'] = (
    W_SOLVENCIA * df['n_renta'].fillna(0) +
    W_VOL       * df['n_vol_75plus'] +
    W_PCT       * df['n_pct_75plus'] +
    W_SOLEDAD   * df['n_ratio_viudez'] +
    W_FUTURO    * df['n_cohorte_futura']
)

# Aplicar filtro duro — secciones no solventes = 0
df.loc[df['solvente'] == 0, 'fofm'] = 0

print("=== DISTRIBUCIÓN DEL ÍNDICE F-of-M ===")
print(f"  Secciones con fofm > 0:   {(df['fofm'] > 0).sum():,}")
print(f"  Secciones con fofm = 0:   {(df['fofm'] == 0).sum():,}")
print(f"\n  Media:   {df[df['fofm']>0]['fofm'].mean():.4f}")
print(f"  Máximo:  {df['fofm'].max():.4f}")
print(f"  Mínimo (>0): {df[df['fofm']>0]['fofm'].min():.4f}")

df[['seccion_censal', 'renta_hogar_2024',
    'vol_75plus', 'ratio_viudez', 'fofm']].sort_values(
    'fofm', ascending=False).head(10)

=== DISTRIBUCIÓN DEL ÍNDICE F-of-M ===
  Secciones con fofm > 0:   24,438
  Secciones con fofm = 0:   11,896

  Media:   0.1639
  Máximo:  0.4469
  Mínimo (>0): 0.0864


,seccion_censal,renta_hogar_2024,vol_75plus,ratio_viudez,fofm
20622,2811501002,83812.0,229.0,4.416667,0.446894
20621,2811501001,83812.0,250.0,3.794118,0.436807
20651,2811502001,83812.0,183.0,3.000000,0.426819
20652,2811502002,83812.0,222.0,3.027027,0.425735
23936,3123202006,33555.0,50.0,25.000000,0.424879
20659,2811502009,83812.0,217.0,2.200000,0.423735
20624,2811501005,83812.0,141.0,2.062500,0.419078
20626,2811501007,83812.0,123.0,2.120000,0.418042
20628,2811501009,83812.0,142.0,1.718750,0.414715
20627,2811501008,83812.0,179.0,1.313725,0.414234


### Paso 4 — Añadir nombres de municipio y crear ranking

In [14]:
# Añadir nombre de municipio
df = df.merge(
    df_muni[['CPRO', 'CMUN', 'NOMBRE']],
    on=['CPRO', 'CMUN'],
    how='left'
)

# Crear ranking
df['ranking'] = df['fofm'].rank(ascending=False, method='min').astype(int)

# Vista del top 20
top20 = df[df['fofm'] > 0].sort_values('fofm', ascending=False).head(20)

print("=== TOP 20 SECCIONES ===")
print(top20[['ranking', 'seccion_censal', 'NOMBRE',
             'renta_hogar_2024', 'vol_75plus',
             'ratio_viudez', 'fofm']].to_string(index=False))

=== TOP 20 SECCIONES ===
 ranking seccion_censal             NOMBRE  renta_hogar_2024  vol_75plus  ratio_viudez     fofm
       1     2811501002 Pozuelo de Alarcón           83812.0       229.0      4.416667 0.446894
       2     2811501001 Pozuelo de Alarcón           83812.0       250.0      3.794118 0.436807
       3     2811502001 Pozuelo de Alarcón           83812.0       183.0      3.000000 0.426819
       4     2811502002 Pozuelo de Alarcón           83812.0       222.0      3.027027 0.425735
       5     3123202006             Tudela           33555.0        50.0     25.000000 0.424879
       6     2811502009 Pozuelo de Alarcón           83812.0       217.0      2.200000 0.423735
       7     2811501005 Pozuelo de Alarcón           83812.0       141.0      2.062500 0.419078
       8     2811501007 Pozuelo de Alarcón           83812.0       123.0      2.120000 0.418042
       9     2811501009 Pozuelo de Alarcón           83812.0       142.0      1.718750 0.414715
      10     28

### Paso 5 — Agregar por municipio
Una residencia capta demanda de varias secciones censales.
Agrupamos por municipio para tener una visión real del mercado.

In [15]:
df_municipios = df[df['fofm'] > 0].groupby(
    ['CPRO', 'CMUN', 'NOMBRE']
).agg(
    secciones          = ('seccion_censal', 'count'),
    fofm_medio         = ('fofm', 'mean'),
    fofm_max           = ('fofm', 'max'),
    vol_75plus_total   = ('vol_75plus', 'sum'),
    pct_75plus_media   = ('pct_75plus', 'mean'),
    ratio_viudez_media = ('ratio_viudez', 'mean'),
    cohorte_futura     = ('cohorte_futura', 'sum'),
    renta_hogar        = ('renta_hogar_2024', 'mean'),
    latitud            = ('latitud', 'mean'),
    longitud           = ('longitud', 'mean')
).reset_index()

# Ranking por municipio
df_municipios = df_municipios.sort_values(
    'fofm_medio', ascending=False
).reset_index(drop=True)

df_municipios['ranking_municipio'] = df_municipios.index + 1

print(f"Municipios con fofm > 0: {df_municipios.shape[0]:,}")
print("\n=== TOP 20 MUNICIPIOS ===")
print(df_municipios[['ranking_municipio', 'NOMBRE', 'renta_hogar',
                      'vol_75plus_total', 'fofm_medio']].head(20).to_string(index=False))

Municipios con fofm > 0: 2,825

=== TOP 20 MUNICIPIOS ===
 ranking_municipio                    NOMBRE  renta_hogar  vol_75plus_total  fofm_medio
                 1        Pozuelo de Alarcón      83812.0            7192.0    0.402604
                 2                Matadepera      68286.0             757.0    0.327867
                 3                  Ergoiena      57526.0              74.0    0.304646
                 4        Boadilla del Monte      65201.0            3011.0    0.302314
                 5               Majadahonda      60937.0            5384.0    0.292940
                 6              Torrelodones      61366.0            1446.0    0.292803
                 7      Rozas de Madrid, Las      61380.0            5571.0    0.288809
                 8            Urdazubi/Urdax      61615.0              36.0    0.287062
                 9       Granja de Moreruela      52541.0              62.0    0.280022
                10                    Laukiz      58362.0     

### Paso 6 — Filtro de tamaño mínimo y selección de 100 ubicaciones

In [16]:
# Filtro mínimo: al menos 500 personas de 75+ en el municipio
# Una residencia de 100 plazas necesita un mercado de captación real
MINIMO_75PLUS = 500

df_candidatos = df_municipios[
    df_municipios['vol_75plus_total'] >= MINIMO_75PLUS
].copy()

print(f"Municipios tras filtro de tamaño: {df_candidatos.shape[0]}")

# Máximo 3 municipios por provincia para diversificar
top100 = (df_candidatos
    .groupby('CPRO')
    .apply(lambda x: x.nlargest(3, 'fofm_medio'))
    .reset_index(drop=True)
    .nlargest(100, 'fofm_medio')
    .reset_index(drop=True)
)

top100['posicion_final'] = top100.index + 1

print(f"Ubicaciones seleccionadas: {len(top100)}")
print(f"Provincias representadas:  {top100['CPRO'].nunique()}")
print(f"\n=== TOP 100 UBICACIONES FINALES ===")
print(top100[['posicion_final', 'NOMBRE', 'CPRO',
              'renta_hogar', 'vol_75plus_total',
              'fofm_medio']].to_string(index=False))

Municipios tras filtro de tamaño: 700
Ubicaciones seleccionadas: 100
Provincias representadas:  45

=== TOP 100 UBICACIONES FINALES ===
 posicion_final                               NOMBRE  CPRO  renta_hogar  vol_75plus_total  fofm_medio
              1                   Pozuelo de Alarcón  28.0      83812.0            7192.0    0.402604
              2                           Matadepera   8.0      68286.0             757.0    0.327867
              3                   Boadilla del Monte  28.0      65201.0            3011.0    0.302314
              4                          Majadahonda  28.0      60937.0            5384.0    0.292940
              5                Sant Cugat del Vallès   8.0      58164.0            5967.0    0.274414
              6                               Alella   8.0      54485.0             945.0    0.270156
              7                                Getxo  48.0      49016.0            9330.0    0.248747
              8                                O

### Paso 7 — Guardar resultados finales

In [17]:
# Guardar el ranking completo de municipios
df_municipios.to_csv('ranking_municipios.csv', index=False, encoding='utf-8-sig')

# Guardar las 100 ubicaciones finales
top100.to_csv('top100_ubicaciones.csv', index=False, encoding='utf-8-sig')

# Resumen ejecutivo por provincia
resumen_provincia = (top100.groupby('CPRO')
    .agg(
        municipios     = ('NOMBRE', 'count'),
        fofm_medio     = ('fofm_medio', 'mean'),
        vol_75plus     = ('vol_75plus_total', 'sum'),
        renta_media    = ('renta_hogar', 'mean')
    )
    .sort_values('fofm_medio', ascending=False)
    .reset_index()
)

resumen_provincia.to_csv('resumen_provincias.csv', index=False, encoding='utf-8-sig')

print("Archivos guardados:")
print(f"  ranking_municipios.csv  → {df_municipios.shape[0]:,} municipios")
print(f"  top100_ubicaciones.csv  → {len(top100)} ubicaciones")
print(f"  resumen_provincias.csv  → {resumen_provincia.shape[0]} provincias")

Archivos guardados:
  ranking_municipios.csv  → 2,825 municipios
  top100_ubicaciones.csv  → 100 ubicaciones
  resumen_provincias.csv  → 45 provincias


In [18]:
# Requiere: pip install openpyxl (ya lo tienes instalado)
with pd.ExcelWriter('resultados_akademia.xlsx', engine='openpyxl') as writer:
    top100.to_excel(writer, sheet_name='Top 100 ubicaciones', index=False)
    df_municipios.head(500).to_excel(writer, sheet_name='Ranking municipios', index=False)
    resumen_provincia.to_excel(writer, sheet_name='Resumen por provincia', index=False)

print("resultados_akademia.xlsx generado")

resultados_akademia.xlsx generado
